# Подготовка данных рейтинга горнолыжных курортов

Тетрадка загружает подготовленные источники, рассчитывает оценки курортов по месяцам и формирует таблицы для анализа и Power BI.

**Основные правила расчёта**

- В рейтинг входят активные курорты и месяцы горнолыжного сезона: ноябрь–май.
- Температурный максимум — от −7 до −5 °C; за пределами диапазона оценка линейно снижается до нуля при −20 и +3 °C.
- Снегопад и снежные дни оцениваются по общей шкале всех курортов и месяцев сезона.
- Стоимость многодневного взрослого или общего ски-пасса переводится в цену за день. Часовые, подъёмные, детские и неоднозначные тарифы исключаются. Ручные дневные цены заменяют тарифный расчёт для соответствующих курортов.
- Погодная, инфраструктурная и ценовая оценки имеют одинаковый вес. Итог — среднее доступных оценок; пропущенная цена не подменяется.
- Инфраструктурная оценка — среднее нормированных длины трасс, числа подъёмников и диапазона высот.
- Данные аэропортов и трансферов используются как справочная информация и в рейтинг не входят.

**Порядок работы:** источники → проверки → сезонные данные → баллы → рейтинг → экспорт → аэропорты и трансферы → итоговые проверки.

Запускать ячейки нужно сверху вниз. Для рейтинга интернет не требуется: используются уже сохранённые исходные CSV-файлы.

## 1. Настройки проекта и пути к файлам

Здесь задаются папки проекта, входные CSV-файлы и места для готовых выгрузок. Проверка останавливает запуск сразу, если обязательного источника нет.


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Ищем корень пакета по структуре data/ и notebooks/; исходная папка проекта не нужна.
PROJECT_DIR = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_DIR = candidate
        break
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Не найдена корневая папка пакета (ожидаются папки data и notebooks). "
        "Откройте тетрадку из папки пакета."
    )

DATA_DIR = PROJECT_DIR / "data"
REFERENCE_DIR = DATA_DIR / "reference"
PROCESSED_DIR = DATA_DIR / "processed"

MASTER_PATH = PROCESSED_DIR / "resort_data_master.csv"
CLIMATE_PATH = PROCESSED_DIR / "weather_monthly_climatology.csv"
PRICES_PATH = REFERENCE_DIR / "resort_prices.csv"
PRICE_ADDITIONS_PATH = PROCESSED_DIR / "powerbi_daily_price_additions.csv"

OUTPUT_FULL_PATH = PROCESSED_DIR / "resort_monthly_ranking.csv"
OUTPUT_RANKING_PATH = PROCESSED_DIR / "powerbi_monthly_ranking.csv"
OUTPUT_RESORTS_PATH = PROCESSED_DIR / "powerbi_resorts.csv"
AIRPORT_ROUTES_PATH = PROCESSED_DIR / "resort_airport_routes_top3.csv"
BEST_AIRPORT_PATH = PROCESSED_DIR / "resort_best_airport_by_road.csv"
TRANSFER_OPTIONS_PATH = REFERENCE_DIR / "resort_transfer_options.csv"
OUTPUT_AIRPORT_ROUTES_PATH = PROCESSED_DIR / "powerbi_airport_routes_top3.csv"
OUTPUT_RESORTS_TRANSFER_PATH = PROCESSED_DIR / "powerbi_resorts_with_transfers.csv"
OUTPUT_TRANSFER_OPTIONS_PATH = PROCESSED_DIR / "powerbi_transfer_options.csv"
OUTPUT_TRANSFER_SUMMARY_PATH = PROCESSED_DIR / "resort_transfer_summary.csv"

REQUIRED_INPUTS = [
    MASTER_PATH, CLIMATE_PATH, PRICES_PATH, PRICE_ADDITIONS_PATH,
    AIRPORT_ROUTES_PATH, BEST_AIRPORT_PATH, TRANSFER_OPTIONS_PATH,
]
missing_files = [path for path in REQUIRED_INPUTS if not path.exists()]
if missing_files:
    raise FileNotFoundError("Не найдены входные файлы:\n" + "\n".join(map(str, missing_files)))

print("Корневая папка пакета:", PROJECT_DIR)
print("Входные файлы найдены:", len(REQUIRED_INPUTS))


Папка проекта: C:\Users\Admin\Documents\Проект Горнолыжки
Входные файлы найдены: 7


## 2. Загрузка исходных таблиц

Загружаются мастер-таблица курортов, климатология, тарифы ски-пассов и дополнения с вручную собранными дневными ценами.


In [2]:
resort_master = pd.read_csv(MASTER_PATH, encoding="utf-8-sig")
climate = pd.read_csv(CLIMATE_PATH, encoding="utf-8-sig")
prices = pd.read_csv(PRICES_PATH, encoding="utf-8-sig")
price_additions = pd.read_csv(PRICE_ADDITIONS_PATH, encoding="utf-8-sig")

print(f"Справочник курортов: {resort_master.shape}")
print(f"Климатология: {climate.shape}")
print(f"Таблица тарифов: {prices.shape}")
print(f"Дополнительные дневные цены: {price_additions.shape}")
display(price_additions)


Справочник курортов: (32, 42)
Климатология: (360, 26)
Таблица тарифов: (526, 20)
Дополнительные дневные цены: (13, 6)


,resort_id,resort_name,avg_daily_ski_pass_price_rub,pricing_unit,visitor_category,data_status
0,R001,Роза Хутор,2100,1 день,не указано,user_provided
1,R002,Красная Поляна,1749,1 день,не указано,user_provided
2,R003,Газпром Поляна,1749,1 день,не указано,user_provided
3,R007,Эльбрус,2200,1 день,не указано,user_provided
4,R009,Мамисон,2200,1 день,не указано,user_provided
5,R018,Морозная гора,1800,1 день,не указано,user_provided
6,R019,Бобровый лог,1377,1 день,не указано,user_provided
7,R023,Аджигардак,2000,1 день,не указано,user_provided
8,R024,Солнечная долина,2500,1 день,не указано,user_provided
9,R025,Завьялиха,3100,1 день,не указано,user_provided


## 3. Проверка справочника и климатологии

Проверяются активный статус курортов, уникальность `resort_id` и отсутствие повторов по паре «курорт–месяц».


In [3]:
active_resorts = resort_master.loc[
    resort_master["project_status"].eq("active")
].copy()

assert active_resorts["resort_id"].is_unique, "В справочнике повторяются resort_id"
assert not climate.duplicated(["resort_id", "month_number"]).any(), \
    "В климатологии есть повторные строки курорт-месяц"
assert set(climate["resort_id"]) >= set(active_resorts["resort_id"]), \
    "Для части активных курортов нет климатологии"

print("Активных курортов:", active_resorts["resort_id"].nunique())
print("Курортов в климатологии:", climate["resort_id"].nunique())
print("Месяцев в климатологии:", climate["month_number"].nunique())
print("Дубликатов курорт-месяц:", climate.duplicated(["resort_id", "month_number"]).sum())


Активных курортов: 30
Курортов в климатологии: 30
Месяцев в климатологии: 12
Дубликатов курорт-месяц: 0


## 4. Объединение данных и выбор горнолыжного сезона

Климатология объединяется с мастер-таблицей по `resort_id`. В рейтинговую выборку входят январь–май и ноябрь–декабрь.


In [4]:
# Название курорта берём из мастер-справочника, чтобы оно было единообразным.
resort_attributes = active_resorts.drop(columns=["resort_name"], errors="ignore")

dashboard_weather_all = climate.merge(
    resort_attributes,
    on="resort_id",
    how="inner",
    validate="many_to_one",
)

dashboard_weather_all["weather_basis"] = "climatology"

SKI_SEASON_MONTHS = [11, 12, 1, 2, 3, 4, 5]
dashboard_weather = dashboard_weather_all.loc[
    dashboard_weather_all["month_number"].isin(SKI_SEASON_MONTHS)
].copy()
dashboard_weather["is_ski_season"] = True

assert not dashboard_weather.duplicated(["resort_id", "month_number"]).any()
print("Сезонные строки:", len(dashboard_weather))
print("Курортов:", dashboard_weather["resort_id"].nunique())
display(
    dashboard_weather.groupby(["month_number", "month_name_ru"])["resort_id"]
    .nunique().rename("курортов").reset_index()
)


Сезонные строки: 210
Курортов: 30


,month_number,month_name_ru,курортов
0,1,январь,30
1,2,февраль,30
2,3,март,30
3,4,апрель,30
4,5,май,30
5,11,ноябрь,30
6,12,декабрь,30


## 5. Погодные баллы


Погодный балл — среднее трёх равновесных компонентов: температурного комфорта, месячного снегопада и числа снежных дней. Снеговые показатели нормируются на максимум среди всех курортов и месяцев сезона.


In [5]:
max_snowfall = dashboard_weather["mean_monthly_snowfall_cm"].max()
max_snowy_days = dashboard_weather["mean_snowy_days_per_month"].max()
if max_snowfall <= 0 or max_snowy_days <= 0:
    raise ValueError("Невозможно рассчитать шкалу снега: максимум должен быть больше нуля")

dashboard_weather["snowfall_score"] = (
    dashboard_weather["mean_monthly_snowfall_cm"] / max_snowfall * 100
)
dashboard_weather["snowy_days_score"] = (
    dashboard_weather["mean_snowy_days_per_month"] / max_snowy_days * 100
)

temp = dashboard_weather["mean_daily_mean_temp_c"]
dashboard_weather["temperature_score"] = np.select(
    [temp.between(-7, -5), temp < -7, temp > -5],
    [
        100,
        ((temp + 20) / 13 * 100).clip(0, 100),
        ((3 - temp) / 8 * 100).clip(0, 100),
    ],
    default=0,
)

weather_components = ["temperature_score", "snowfall_score", "snowy_days_score"]
dashboard_weather["weather_score"] = dashboard_weather[weather_components].mean(axis=1)

print("Предел шкалы снегопада:", round(max_snowfall, 2), "см")
print("Предел шкалы снежных дней:", round(max_snowy_days, 1))
display(
    dashboard_weather.loc[dashboard_weather["month_number"].eq(12)]
    .sort_values("weather_score", ascending=False)[
        ["resort_name", "mean_daily_mean_temp_c", "mean_monthly_snowfall_cm",
         "mean_snowy_days_per_month", "temperature_score", "weather_score"]
    ].head(10)
)


Предел шкалы снегопада: 101.56 см
Предел шкалы снежных дней: 21.8


,resort_name,mean_daily_mean_temp_c,mean_monthly_snowfall_cm,mean_snowy_days_per_month,temperature_score,weather_score
71,Домбай,-2.79,92.37,17.9,72.375000,81.812085
95,Чегет,-6.53,68.08,16.6,100.000000,81.060351
203,Морозная гора,-10.24,83.38,16.9,75.076923,78.233037
59,Архыз,-0.24,96.22,18.8,40.500000,73.826852
155,Большой Вудъявр,-9.66,38.50,21.8,79.538462,72.482362
83,Эльбрус,-4.85,45.20,14.9,98.125000,70.326445
347,Сорочаны,-4.29,32.12,19.1,91.125000,70.122101
167,Кукисвумчорр,-9.73,32.40,20.4,79.000000,68.160102
335,Игора,-3.83,29.74,18.5,85.375000,66.506856
323,Губаха,-11.78,42.99,20.3,63.230769,66.226564


## 6. Средняя цена ски-пасса за день


В расчёт попадают взрослые или общие тарифы в рублях с распознанным числом дней катания. Цена за весь период делится на дни. Тарифы в часах, подъёмах и сезонные абонементы не переводятся в цену за день. Для курортов с вручную внесённой дневной ценой она заменяет результат по тарифам.


In [6]:
ski_prices = prices.loc[prices["service_category"].eq("ski_pass")].copy()
ski_prices["price_amount"] = pd.to_numeric(ski_prices["price_amount"], errors="coerce")

def parse_ski_days(unit):
    if pd.isna(unit):
        return np.nan
    text = str(unit).strip().lower()
    if "season" in text or "сезон" in text:
        return np.nan
    match = re.search(r"(\d+)\s*(?:день|дня|дней|days?)", text)
    if match:
        return int(match.group(1))
    if text in {"день", "day"}:
        return 1
    return np.nan

excluded_terms = (
    "дет", "child", "семь", "льгот", "пенсион", "школь",
    "спортсмен", "тренер", "инвалид", "студент", "14–24",
    "14-24", "65+", "гость курортного размещения",
)

def is_general_or_adult(value):
    if pd.isna(value):
        return True
    text = str(value).strip().lower().replace("ё", "е")
    return not any(term in text for term in excluded_terms)

ski_prices["ski_days"] = ski_prices["pricing_unit"].apply(parse_ski_days)
ski_prices["adult_or_general"] = ski_prices["visitor_category"].apply(is_general_or_adult)

daily_rows = ski_prices.loc[
    ski_prices["currency"].eq("RUB")
    & ski_prices["ski_days"].notna()
    & ski_prices["adult_or_general"]
    & ski_prices["price_amount"].gt(0)
].copy()
daily_rows["price_per_ski_day_rub"] = daily_rows["price_amount"] / daily_rows["ski_days"]

price_daily = (
    daily_rows.groupby("resort_id")
    .agg(
        avg_daily_ski_pass_price_rub=("price_per_ski_day_rub", "mean"),
        median_daily_ski_pass_price_rub=("price_per_ski_day_rub", "median"),
        min_daily_ski_pass_price_rub=("price_per_ski_day_rub", "min"),
        max_daily_ski_pass_price_rub=("price_per_ski_day_rub", "max"),
        price_tariff_count=("price_per_ski_day_rub", "count"),
    )
)

# Заменяем автоматические значения на сообщённые вручную дневные цены.
manual_prices = price_additions.set_index("resort_id")
for column in [
    "avg_daily_ski_pass_price_rub", "median_daily_ski_pass_price_rub",
    "min_daily_ski_pass_price_rub", "max_daily_ski_pass_price_rub",
]:
    manual_prices[column] = pd.to_numeric(
        manual_prices["avg_daily_ski_pass_price_rub"], errors="coerce"
    )
manual_prices["price_tariff_count"] = 1
manual_prices = manual_prices[
    ["avg_daily_ski_pass_price_rub", "median_daily_ski_pass_price_rub",
     "min_daily_ski_pass_price_rub", "max_daily_ski_pass_price_rub",
     "price_tariff_count"]
]
price_daily = price_daily.drop(index=manual_prices.index, errors="ignore")
price_daily = pd.concat([price_daily, manual_prices]).reset_index()

dashboard_weather = dashboard_weather.merge(
    price_daily, on="resort_id", how="left", validate="many_to_one"
)

print("Тарифов, переведённых в цену за день:", len(daily_rows))
print("Курортов с ценой за день:", price_daily["resort_id"].nunique())
display(
    price_daily.merge(
        active_resorts[["resort_id", "resort_name"]],
        on="resort_id", how="left", validate="one_to_one"
    ).sort_values("avg_daily_ski_pass_price_rub")
    [["resort_name", "avg_daily_ski_pass_price_rub", "price_tariff_count"]]
)


Тарифов, переведённых в цену за день: 83
Курортов с ценой за день: 30


,resort_name,avg_daily_ski_pass_price_rub,price_tariff_count
4,Цей,600.000000,1
23,Бобровый лог,1377.000000,1
6,Телецкий,1643.750000,8
8,Кукисвумчорр,1700.000000,2
18,Красная Поляна,1749.000000,1
19,Газпром Поляна,1749.000000,1
1,Архыз,1800.000000,1
22,Морозная гора,1800.000000,1
29,Красная Глинка,1900.000000,1
7,Большой Вудъявр,1942.133333,20


## 7. Баллы за инфраструктуру и цену


Инфраструктурный балл — среднее трёх оценок по шкале min–max: длина трасс, число подъёмников и диапазон высот. Ценовой балл выше при меньшей средней цене катания за день.


In [7]:
def minmax_score(series):
    values = pd.to_numeric(series, errors="coerce")
    low, high = values.min(), values.max()
    if pd.isna(low) or pd.isna(high) or high == low:
        return pd.Series(np.nan, index=series.index, dtype="float64")
    return (values - low) / (high - low) * 100

# Инфраструктура считается на уровне курорта, поэтому сначала используем
# уникальные значения курортов и затем присоединяем баллы к месячной таблице.
infrastructure = active_resorts[[
    "resort_id", "piste_total_km", "lift_count", "elevation_range_openskidata_m"
]].drop_duplicates("resort_id").copy()
infrastructure["piste_length_score"] = minmax_score(infrastructure["piste_total_km"])
infrastructure["lift_count_score"] = minmax_score(infrastructure["lift_count"])
infrastructure["elevation_range_score"] = minmax_score(
    infrastructure["elevation_range_openskidata_m"]
)
infrastructure_parts = ["piste_length_score", "lift_count_score", "elevation_range_score"]
infrastructure["infrastructure_score"] = infrastructure[infrastructure_parts].mean(axis=1)

dashboard_weather = dashboard_weather.merge(
    infrastructure[["resort_id", *infrastructure_parts, "infrastructure_score"]],
    on="resort_id", how="left", validate="many_to_one"
)

price_min = price_daily["avg_daily_ski_pass_price_rub"].min()
price_max = price_daily["avg_daily_ski_pass_price_rub"].max()
if price_max == price_min:
    dashboard_weather["price_score"] = np.nan
else:
    dashboard_weather["price_score"] = (
        100 * (price_max - dashboard_weather["avg_daily_ski_pass_price_rub"])
        / (price_max - price_min)
    )

print("Шкала цены:", round(price_min, 2), "—", round(price_max, 2), "руб./день")


Шкала цены: 600.0 — 3916.0 руб./день


## 8. Итоговая оценка и месячный ранг


In [8]:
rating_criteria = ["weather_score", "infrastructure_score", "price_score"]
dashboard_weather["criteria_used_count"] = dashboard_weather[rating_criteria].notna().sum(axis=1)
dashboard_weather["overall_score"] = dashboard_weather[rating_criteria].mean(axis=1, skipna=True)

dashboard_weather["overall_rank"] = (
    dashboard_weather.groupby("month_number")["overall_score"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
dashboard_weather["weather_rank"] = (
    dashboard_weather.groupby("month_number")["weather_score"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

print("Строк рейтинга:", len(dashboard_weather))
print("Критериев использовано (распределение):")
display(dashboard_weather["criteria_used_count"].value_counts().sort_index())

selected_month = 12  # номер месяца для примера: 1–5, 11 или 12
month_example = (
    dashboard_weather.loc[dashboard_weather["month_number"].eq(selected_month)]
    .sort_values("overall_rank")
)
display(month_example[[
    "overall_rank", "resort_name", "overall_score", "weather_score",
    "infrastructure_score", "price_score", "criteria_used_count"
]].head(15))


Строк рейтинга: 210
Критериев использовано (распределение):


3    210
Name: criteria_used_count, dtype: int64

,overall_rank,resort_name,overall_score,weather_score,infrastructure_score,price_score,criteria_used_count
6,1,Роза Хутор,69.508602,54.400737,99.360292,54.764777,3
20,2,Газпром Поляна,63.219708,54.400737,69.908569,65.349819,3
34,3,Архыз,61.625465,73.826852,47.237722,63.811821,3
69,4,Цей,58.281944,65.698167,9.147664,100.000000,3
27,5,Шерегеш,58.112530,58.318434,69.407890,46.611267,3
13,6,Красная Поляна,57.876458,54.400737,53.878819,65.349819,3
48,7,Эльбрус,56.527721,70.326445,47.507623,51.749095,3
90,8,Большой Вудъявр,56.363976,72.482362,37.084033,59.525533,3
76,9,Манжерок,52.587519,59.793628,50.743354,47.225573,3
118,10,Морозная гора,52.362913,78.233037,15.043881,63.811821,3


## 9. Экспорт рейтинга и справочника курортов


`resort_monthly_ranking.csv` содержит полный расчёт. `powerbi_monthly_ranking.csv` — компактная таблица «курорт–месяц». `powerbi_resorts.csv` — справочник активных курортов. Дополнительные справочные выгрузки по аэропортам и трансферам формируются ниже.


In [9]:
# Полная рабочая месячная таблица
dashboard_weather.to_csv(OUTPUT_FULL_PATH, index=False, encoding="utf-8-sig")

ranking_columns = [
    "resort_id", "resort_name", "month_number", "month_name_ru",
    "overall_rank", "overall_score", "criteria_used_count", "weather_rank",
    "weather_score", "temperature_score", "snowfall_score", "snowy_days_score",
    "mean_daily_mean_temp_c", "mean_monthly_snowfall_cm",
    "mean_snowy_days_per_month", "avg_daily_ski_pass_price_rub",
    "median_daily_ski_pass_price_rub", "min_daily_ski_pass_price_rub",
    "max_daily_ski_pass_price_rub", "price_tariff_count", "price_score",
    "infrastructure_score", "piste_length_score", "lift_count_score",
    "elevation_range_score", "weather_basis", "is_ski_season",
]
powerbi_ranking = dashboard_weather[ranking_columns].copy()
powerbi_ranking.to_csv(OUTPUT_RANKING_PATH, index=False, encoding="utf-8-sig")

resort_columns = [
    "resort_id", "resort_name", "region", "resort_official_url",
    "latitude", "longitude", "openskidata_name", "piste_total_km",
    "lift_count", "elevation_min_m", "elevation_max_m",
    "elevation_range_openskidata_m", "activity_tags",
    "piste_difficulty_categories", "openskidata_source_url", "piste_source_url",
]
resort_columns = [column for column in resort_columns if column in active_resorts.columns]
powerbi_resorts = active_resorts[resort_columns].drop_duplicates("resort_id").copy()
powerbi_resorts.to_csv(OUTPUT_RESORTS_PATH, index=False, encoding="utf-8-sig")

print("Сохранены файлы:")
for path in [OUTPUT_FULL_PATH, OUTPUT_RANKING_PATH, OUTPUT_RESORTS_PATH]:
    print("-", path)


Сохранены файлы:
- C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\resort_monthly_ranking.csv
- C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\powerbi_monthly_ranking.csv
- C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\powerbi_resorts.csv


## 10. Проверки рейтинга

Проверяется число строк, наличие всех курортов в каждом месяце, уникальность пары «курорт–месяц» и заполненность итогового балла.


In [10]:
expected_rows = active_resorts["resort_id"].nunique() * len(SKI_SEASON_MONTHS)
checks = {
    "Число строк рейтинга соответствует курортам × месяцам": len(powerbi_ranking) == expected_rows,
    "В каждом месяце присутствуют все активные курорты": (
        powerbi_ranking.groupby("month_number")["resort_id"].nunique().eq(
            active_resorts["resort_id"].nunique()
        ).all()
    ),
    "Нет повторов курорт-месяц": not powerbi_ranking.duplicated(
        ["resort_id", "month_number"]
    ).any(),
    "Нет пропусков итогового балла": powerbi_ranking["overall_score"].notna().all(),
}
for description, passed in checks.items():
    print(("✓" if passed else "✗"), description)

if not all(checks.values()):
    raise ValueError("Одна или несколько итоговых проверок не пройдены")

print("\nСтрок рейтинга:", len(powerbi_ranking))
print("Курортов:", powerbi_ranking["resort_id"].nunique())
print("Месяцев:", powerbi_ranking["month_number"].nunique())
print("Курортов с ценовым баллом:", powerbi_ranking.loc[
    powerbi_ranking["price_score"].notna(), "resort_id"
].nunique())


✓ Число строк рейтинга соответствует курортам × месяцам
✓ В каждом месяце присутствуют все активные курорты
✓ Нет повторов курорт-месяц
✓ Нет пропусков итогового балла

Строк рейтинга: 210
Курортов: 30
Месяцев: 7
Курортов с ценовым баллом: 30


## 11. Аэропорты и варианты трансфера

Здесь используются уже собранные файлы с маршрутами и трансферами. Сетевые запросы и ручное добавление справочных строк в чистовой тетрадке не выполняются: источником истины остаётся CSV в `data/reference`.

### 11.1 Загрузка и проверка справочных данных

`resort_airport_routes_top3.csv` содержит до трёх аэропортов-кандидатов на курорт. `resort_best_airport_by_road.csv` содержит выбранный вариант с кратчайшим расчётным временем по дороге. Статус авиасообщения хранится отдельно: ближайший аэропорт не обязательно обслуживает регулярные рейсы.

`resort_transfer_options.csv` хранит отдельные предложения перевозчиков и общественного транспорта. Цены могут быть за человека или за автомобиль, поэтому ниже они группируются только внутри одинаковых единиц тарификации.

In [11]:
airport_routes = pd.read_csv(AIRPORT_ROUTES_PATH, encoding="utf-8-sig")
best_airports = pd.read_csv(BEST_AIRPORT_PATH, encoding="utf-8-sig")
transfer_options = pd.read_csv(TRANSFER_OPTIONS_PATH, encoding="utf-8-sig")
transfer_options["fare_amount"] = pd.to_numeric(
    transfer_options["fare_amount"], errors="coerce"
)

assert best_airports["resort_id"].is_unique, "В таблице лучших аэропортов есть повторы resort_id"
assert set(best_airports["resort_id"]) == set(active_resorts["resort_id"]),     "Набор курортов в таблице лучших аэропортов не совпадает с активным справочником"
assert set(airport_routes["resort_id"]) == set(active_resorts["resort_id"]),     "В таблице маршрутов отсутствуют активные курорты"
assert set(transfer_options["resort_id"]) == set(active_resorts["resort_id"]),     "В таблице трансферов отсутствуют активные курорты"

print("Кандидатов аэропортов:", len(airport_routes))
print("Выбранных аэропортов:", len(best_airports))
print("Вариантов трансфера:", len(transfer_options))
print("Курортов с вариантами трансфера:", transfer_options["resort_id"].nunique())

Кандидатов аэропортов: 90
Выбранных аэропортов: 30
Вариантов трансфера: 43
Курортов с вариантами трансфера: 30


### 11.2 Сводка трансферов по курортам

Сводка содержит одну строку на курорт: количество вариантов, аэропорты, типы услуг, число тарифов с ценой и текстовое представление цен по единице тарификации. Числа в сводке не являются общей оценкой стоимости поездки.

In [12]:
def summarize_fares(group):
    priced = group.loc[group["fare_amount"].notna()]
    if priced.empty:
        return "нет опубликованных цен"

    parts = []
    for (currency, unit), fares in priced.groupby(
        ["currency", "fare_unit"], dropna=False
    ):
        low = fares["fare_amount"].min()
        high = fares["fare_amount"].max()
        amount = f"{low:g}" if low == high else f"{low:g}–{high:g}"
        parts.append(f"{amount} {currency or ''} ({unit or 'единица не указана'})")
    return "; ".join(parts)

summary_rows = []
for (resort_id, resort_name), group in transfer_options.groupby(
    ["resort_id", "resort_name"], dropna=False, sort=True
):
    summary_rows.append({
        "resort_id": resort_id,
        "resort_name": resort_name,
        "transfer_options_count": len(group),
        "airport_codes": ", ".join(sorted(group["airport_iata_code"].dropna().astype(str).unique())),
        "service_types": ", ".join(sorted(group["service_type"].dropna().astype(str).unique())),
        "fares_published_count": group["fare_amount"].notna().sum(),
        "fares_without_price_count": group["fare_amount"].isna().sum(),
        "fare_summary_by_unit": summarize_fares(group),
        "candidate_count": group["data_status"].eq("candidate").sum(),
        "source_checked_count": group["data_status"].eq("source_checked").sum(),
    })

transfer_summary = pd.DataFrame(summary_rows)
assert transfer_summary["resort_id"].is_unique
assert len(transfer_summary) == active_resorts["resort_id"].nunique()
transfer_summary.to_csv(
    OUTPUT_TRANSFER_SUMMARY_PATH, index=False, encoding="utf-8-sig"
)

display(transfer_summary.head(10))
print("Сводка сохранена:", OUTPUT_TRANSFER_SUMMARY_PATH)

,resort_id,resort_name,transfer_options_count,airport_codes,service_types,fares_published_count,fares_without_price_count,fare_summary_by_unit,candidate_count,source_checked_count
0,R001,Роза Хутор,5,AER,"private_transfer, scheduled_bus, taxi",4,1,1200 RUB (from_per_vehicle); 4100 RUB (from_pe...,4,1
1,R002,Красная Поляна,2,AER,private_transfer,2,0,3600 RUB (from_per_vehicle_one_way),2,0
2,R003,Газпром Поляна,4,AER,hotel_guest_transfer,4,0,2200–2400 RUB (per_vehicle_one_way),0,4
3,R004,Шерегеш,3,NOZ,"private_car, scheduled_bus, seasonal_express_bus",2,1,664 RUB (per_person_one_way); 7000 RUB (per_ve...,2,1
4,R005,Архыз,2,MRV,scheduled_bus,2,0,1200–1300 RUB (per_person_one_way),2,0
5,R006,Домбай,1,MRV,seasonal_bus,1,0,1600 RUB (per_person),0,1
6,R007,Эльбрус,1,MRV,private_transfer,1,0,6000 RUB (per_vehicle),1,0
7,R008,Чегет,1,NAL,private_transfer,1,0,4000 RUB (per_vehicle_one_way),1,0
8,R009,Мамисон,1,,scheduled_shuttle,1,0,200 RUB (not_specified),1,0
9,R010,Цей,2,OGZ,private_transfer,2,0,3400–6700 RUB (per_vehicle_one_way),0,2


Сводка сохранена: C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\resort_transfer_summary.csv


### 11.3 Справочник курортов с аэропортом и сводкой трансферов

К базовой таблице курортов добавляются выбранный аэропорт, дорожные расстояние и время, а также сводка предложений. Связь выполняется по уникальному `resort_id`; число строк справочника не должно увеличиться.

In [13]:
best_airport_columns = [
    "resort_id", "airport_rank_by_straight_line", "airport_iata_code",
    "airport_name", "airport_municipality", "airport_scheduled_service",
    "straight_line_distance_km", "road_distance_km", "road_duration_hours",
    "route_status", "route_source", "source_url", "checked_at_utc",
    "airport_service_verification_status", "airport_service_source_url",
]
best_airports_for_merge = best_airports[best_airport_columns].copy()
summary_for_merge = transfer_summary.drop(columns="resort_name")

powerbi_resorts_with_transfers = powerbi_resorts.merge(
    best_airports_for_merge, on="resort_id", how="left",
    validate="one_to_one", indicator="_airport_match"
)
assert powerbi_resorts_with_transfers["_airport_match"].eq("both").all(),     "Не для всех курортов найден аэропортовый маршрут"
powerbi_resorts_with_transfers = powerbi_resorts_with_transfers.drop(
    columns="_airport_match"
).merge(
    summary_for_merge, on="resort_id", how="left",
    validate="one_to_one", indicator="_transfer_match"
)
assert powerbi_resorts_with_transfers["_transfer_match"].eq("both").all(),     "Не для всех курортов найдена сводка трансферов"
powerbi_resorts_with_transfers = powerbi_resorts_with_transfers.drop(
    columns="_transfer_match"
)
assert len(powerbi_resorts_with_transfers) == len(powerbi_resorts)

powerbi_resorts_with_transfers.to_csv(
    OUTPUT_RESORTS_TRANSFER_PATH, index=False, encoding="utf-8-sig"
)
print("Курортов:", len(powerbi_resorts_with_transfers))
print("Справочник сохранён:", OUTPUT_RESORTS_TRANSFER_PATH)
display(powerbi_resorts_with_transfers[[
    "resort_id", "resort_name", "airport_iata_code", "road_distance_km",
    "road_duration_hours", "transfer_options_count", "fares_published_count",
]].head(10))

Курортов: 30
Справочник сохранён: C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\powerbi_resorts_with_transfers.csv


,resort_id,resort_name,airport_iata_code,road_distance_km,road_duration_hours,transfer_options_count,fares_published_count
0,R001,Роза Хутор,AER,44.3,0.69,5,4
1,R002,Красная Поляна,AER,36.6,0.63,2,2
2,R003,Газпром Поляна,AER,43.4,0.71,4,4
3,R004,Шерегеш,NOZ,177.5,2.83,3,2
4,R005,Архыз,MRV,211.9,3.51,2,2
5,R006,Домбай,MRV,200.3,3.44,1,1
6,R007,Эльбрус,NAL,116.9,2.01,1,1
7,R008,Чегет,NAL,127.1,2.18,1,1
8,R009,Мамисон,OGZ,111.0,1.72,1,1
9,R010,Цей,OGZ,105.2,2.08,2,2


### 11.4 Таблицы маршрутов и вариантов трансфера

Таблицы с несколькими вариантами на курорт сохраняются отдельно. Это позволяет посмотреть альтернативные аэропорты и перевозчиков, не размножая строки месячного рейтинга.

In [14]:
airport_routes.to_csv(
    OUTPUT_AIRPORT_ROUTES_PATH, index=False, encoding="utf-8-sig"
)

transfer_columns = [
    "resort_id", "resort_name", "airport_iata_code", "airport_name",
    "service_type", "provider_name", "origin", "destination", "fare_amount",
    "currency", "fare_unit", "vehicle_capacity", "season", "valid_from",
    "valid_to", "data_status", "checked_at", "source_url", "notes",
]
powerbi_transfer_options = transfer_options[transfer_columns].copy()
powerbi_transfer_options.to_csv(
    OUTPUT_TRANSFER_OPTIONS_PATH, index=False, encoding="utf-8-sig"
)

print("Маршруты аэропортов:", OUTPUT_AIRPORT_ROUTES_PATH)
print("Варианты трансфера:", OUTPUT_TRANSFER_OPTIONS_PATH)
print("Строк маршрутов:", len(airport_routes))
print("Строк вариантов трансфера:", len(powerbi_transfer_options))
print("Вариантов без опубликованной цены:", powerbi_transfer_options["fare_amount"].isna().sum())

Маршруты аэропортов: C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\powerbi_airport_routes_top3.csv
Варианты трансфера: C:\Users\Admin\Documents\Проект Горнолыжки\data\processed\powerbi_transfer_options.csv
Строк маршрутов: 90
Строк вариантов трансфера: 43
Вариантов без опубликованной цены: 6


## 12. Проверка набора для Power BI

В Power BI справочник `powerbi_resorts_with_transfers.csv` связывается по `resort_id` со стороны «один» с тремя таблицами со стороны «многие»: рейтингом по месяцам, маршрутами аэропортов и вариантами трансфера. Не связывайте таблицы фактов друг с другом напрямую.

In [15]:
powerbi_exports = {
    "Рейтинг по месяцам": (OUTPUT_RANKING_PATH, 210),
    "Курорты с аэропортами и трансферами": (OUTPUT_RESORTS_TRANSFER_PATH, 30),
    "Маршруты до аэропортов": (OUTPUT_AIRPORT_ROUTES_PATH, 90),
    "Варианты трансфера": (OUTPUT_TRANSFER_OPTIONS_PATH, 43),
}

for name, (path, expected_rows) in powerbi_exports.items():
    exported = pd.read_csv(path, encoding="utf-8-sig")
    print(
        f"{name}: {len(exported)} строк (ожидалось {expected_rows}), "
        f"{len(exported.columns)} столбцов, файл существует: {path.exists()}"
    )
    assert len(exported) == expected_rows, f"Неверное число строк: {name}"

assert powerbi_resorts_with_transfers["resort_id"].is_unique
assert not powerbi_ranking.duplicated(["resort_id", "month_number"]).any()
assert powerbi_transfer_options["resort_id"].nunique() == 30
print("Проверки набора для Power BI пройдены.")

Рейтинг по месяцам: 210 строк (ожидалось 210), 27 столбцов, файл существует: True
Курорты с аэропортами и трансферами: 30 строк (ожидалось 30), 38 столбцов, файл существует: True
Маршруты до аэропортов: 90 строк (ожидалось 90), 17 столбцов, файл существует: True
Варианты трансфера: 43 строк (ожидалось 43), 19 столбцов, файл существует: True
Проверки набора для Power BI пройдены.
